<a href="https://colab.research.google.com/github/Aasish357/-Cybersecurity-Intelligence-Agents-/blob/main/fraud_reduction_(ML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install imbalanced-learn fastapi uvicorn joblib
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score

from imblearn.over_sampling import SMOTE

import joblib



In [7]:
# Creating a synthetic, imbalanced dataset
# 2 classes, but class 1 (fraud) is rare

X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    weights=[0.98, 0.02],   # 2% fraud
    random_state=42
)

# Put into a DataFrame (easier to see)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
data = pd.DataFrame(X, columns=feature_names)
data["fraud"] = y

data.head()



,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,fraud
0,-1.001394,0.786597,0.331109,0.024749,-1.311850,-1.094704,-1.563117,-0.305194,0.252958,-0.962313,...,-0.306860,0.749424,0.148490,1.415729,-1.944211,0.249360,0.581752,0.474943,-0.060529,0
1,-0.223332,0.704420,-1.433255,2.315630,-0.428886,0.886194,0.481507,-0.067329,-0.587163,3.025559,...,1.869915,2.418724,-1.236955,-0.557634,-2.539610,1.334627,-0.181978,2.470788,-2.309603,0
2,1.180982,-0.793879,-0.037482,0.808142,-1.582501,-0.921351,3.165995,-0.382410,-0.478582,0.620644,...,-0.543322,0.259772,0.859113,-2.364552,0.202232,0.084659,-0.519546,-0.739941,0.711417,0
3,0.755648,1.189717,-1.076123,-0.214387,0.504537,0.543859,-0.041364,0.430982,-0.766390,0.310126,...,-1.174784,1.054988,-0.322950,0.051608,-1.209274,-0.136638,-0.162811,-0.257430,-0.600781,1
4,-0.330034,-0.261929,1.142192,-0.414380,-0.348382,-0.171059,1.465617,-0.354117,1.837575,0.518120,...,0.104226,-1.933259,0.501168,0.281280,-0.064471,1.321861,-0.730852,-1.065595,0.043229,0


In [8]:
# Check how many fraud vs non-fraud cases
class_counts = data["fraud"].value_counts()
print(class_counts)
print("\nClass distribution (%):")
print(class_counts / len(data) * 100)


fraud
0    9744
1     256
Name: count, dtype: int64

Class distribution (%):
fraud
0    97.44
1     2.56
Name: count, dtype: float64


In [9]:
X = data.drop("fraud", axis=1)
y = data["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (8000, 20)
Test shape: (2000, 20)


In [10]:
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")


Scaling complete.


In [11]:
baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(X_train_scaled, y_train)

y_pred_baseline = baseline_model.predict(X_test_scaled)

print("=== Baseline Logistic Regression (No SMOTE) ===")
print(classification_report(y_test, y_pred_baseline))

baseline_cm = confusion_matrix(y_test, y_pred_baseline)
print("\nConfusion Matrix:")
print(baseline_cm)

baseline_recall = recall_score(y_test, y_pred_baseline)
print("\nBaseline Recall (fraud class):", baseline_recall)


=== Baseline Logistic Regression (No SMOTE) ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1949
           1       1.00      0.02      0.04        51

    accuracy                           0.97      2000
   macro avg       0.99      0.51      0.51      2000
weighted avg       0.98      0.97      0.96      2000


Confusion Matrix:
[[1949    0]
 [  50    1]]

Baseline Recall (fraud class): 0.0196078431372549


In [13]:
# Oversampling only on training set
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE :", np.bincount(y_train_smote))


Before SMOTE: [7795  205]
After SMOTE : [7795 7795]


In [14]:
logreg_smote_model = LogisticRegression(max_iter=1000)

logreg_smote_model.fit(X_train_smote, y_train_smote)

y_pred_logreg_smote = logreg_smote_model.predict(X_test_scaled)

print("=== Logistic Regression with SMOTE ===")
print(classification_report(y_test, y_pred_logreg_smote))

logreg_smote_cm = confusion_matrix(y_test, y_pred_logreg_smote)
print("\nConfusion Matrix:")
print(logreg_smote_cm)

logreg_smote_recall = recall_score(y_test, y_pred_logreg_smote)
print("\nRecall (fraud class) with SMOTE:", logreg_smote_recall)


=== Logistic Regression with SMOTE ===
              precision    recall  f1-score   support

           0       0.99      0.71      0.83      1949
           1       0.06      0.69      0.11        51

    accuracy                           0.71      2000
   macro avg       0.52      0.70      0.47      2000
weighted avg       0.96      0.71      0.81      2000


Confusion Matrix:
[[1380  569]
 [  16   35]]

Recall (fraud class) with SMOTE: 0.6862745098039216


In [15]:
# False negatives are: actual=1 (fraud), predicted=0 (non-fraud)
baseline_fn = baseline_cm[1, 0]
smote_fn = logreg_smote_cm[1, 0]

reduction = baseline_fn - smote_fn
if baseline_fn > 0:
    reduction_percent = reduction / baseline_fn * 100
else:
    reduction_percent = 0

print("Baseline False Negatives:", baseline_fn)
print("SMOTE False Negatives   :", smote_fn)
print("Reduction in FN         :", reduction)
print("Reduction (%)           :", round(reduction_percent, 2), "%")


Baseline False Negatives: 50
SMOTE False Negatives   : 16
Reduction in FN         : 34
Reduction (%)           : 68.0 %


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

y_pred_rf = rf_model.predict(X_test_scaled)

print("=== Random Forest with SMOTE ===")
print(classification_report(y_test, y_pred_rf))

rf_cm = confusion_matrix(y_test, y_pred_rf)
print("\nConfusion Matrix:")
print(rf_cm)

rf_recall = recall_score(y_test, y_pred_rf)
print("\nRecall (fraud class) RF:", rf_recall)


=== Random Forest with SMOTE ===
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1949
           1       0.46      0.31      0.37        51

    accuracy                           0.97      2000
   macro avg       0.72      0.65      0.68      2000
weighted avg       0.97      0.97      0.97      2000


Confusion Matrix:
[[1930   19]
 [  35   16]]

Recall (fraud class) RF: 0.3137254901960784


In [17]:
print("LogReg + SMOTE Recall:", logreg_smote_recall)
print("RandomForest Recall  :", rf_recall)

# For simplicity, let's choose the model with higher recall
if rf_recall >= logreg_smote_recall:
    best_model = rf_model
    best_model_name = "random_forest"
else:
    best_model = logreg_smote_model
    best_model_name = "logistic_regression"

print("\nSelected best model:", best_model_name)

# Save the best model and scaler (for API use)
joblib.dump(best_model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved.")


LogReg + SMOTE Recall: 0.6862745098039216
RandomForest Recall  : 0.3137254901960784

Selected best model: logistic_regression
Model and scaler saved.


In [18]:
#now using  Fast API

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Load model and scaler
loaded_model = joblib.load("fraud_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")

app = FastAPI(title="Fraud Detection API")

class Transaction(BaseModel):
    features: list  # expecting a list of 20 feature values

@app.get("/")
def home():
    return {"message": "Fraud detection API is running."}

@app.post("/predict")
def predict_fraud(transaction: Transaction):
    # Convert list to numpy array
    x = np.array(transaction.features).reshape(1, -1)
    # Scale using the same scaler used during training
    x_scaled = loaded_scaler.transform(x)
    # Predict
    prediction = loaded_model.predict(x_scaled)[0]
    # For demo, also return probability if supported
    if hasattr(loaded_model, "predict_proba"):
        prob = float(loaded_model.predict_proba(x_scaled)[0][1])
    else:
        prob = None

    return {
        "fraud_prediction": int(prediction),
        "fraud_probability": prob
    }
